In [ ]:
import time
start_time = time.time()

import pandas as pd
import torch
import torch.nn.functional as F
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix


In [ ]:
device = torch.device("mps") if torch.backends.mps.is_available() else torch.device("cpu")
print(f"Selected device: {device}")

model_name = "sentence-transformers/all-MiniLM-L6-v2"
batch_size = 128
max_length = 128
similarity_threshold = 0.80

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)
model.to(device)
model.eval()

print(f"Loaded model: {model_name}")
print(f"Batch size: {batch_size}")
print(f"Max length: {max_length}")
print(f"Similarity threshold: {similarity_threshold}")


In [ ]:
dataset = load_dataset("glue", "mrpc", split="validation")
print(f"Validation examples: {len(dataset)}")

preview_df = dataset.select(range(min(5, len(dataset)))).to_pandas()
print(preview_df[["sentence1", "sentence2", "label"]].to_string(index=False))


In [ ]:
def mean_pooling(model_output, attention_mask):
    token_embeddings = model_output.last_hidden_state
    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    return torch.sum(token_embeddings * input_mask_expanded, dim=1) / torch.clamp(input_mask_expanded.sum(dim=1), min=1e-9)

predictions = []
true_labels = []
similarities = []

for start_idx in range(0, len(dataset), batch_size):
    batch = dataset[start_idx:start_idx + batch_size]

    inputs_1 = tokenizer(
        batch["sentence1"],
        truncation=True,
        padding=True,
        max_length=max_length,
        return_tensors="pt"
    )
    inputs_2 = tokenizer(
        batch["sentence2"],
        truncation=True,
        padding=True,
        max_length=max_length,
        return_tensors="pt"
    )

    inputs_1 = {k: v.to(device) for k, v in inputs_1.items()}
    inputs_2 = {k: v.to(device) for k, v in inputs_2.items()}

    with torch.no_grad():
        outputs_1 = model(**inputs_1)
        outputs_2 = model(**inputs_2)

        embeddings_1 = mean_pooling(outputs_1, inputs_1["attention_mask"])
        embeddings_2 = mean_pooling(outputs_2, inputs_2["attention_mask"])

        embeddings_1 = F.normalize(embeddings_1, p=2, dim=1)
        embeddings_2 = F.normalize(embeddings_2, p=2, dim=1)

        batch_similarities = torch.sum(embeddings_1 * embeddings_2, dim=1)
        batch_preds = (batch_similarities >= similarity_threshold).long()

    similarities.extend(batch_similarities.cpu().tolist())
    predictions.extend(batch_preds.cpu().tolist())
    true_labels.extend(batch["label"])

print(f"Completed inference for {len(predictions)} examples.")


In [ ]:
accuracy = accuracy_score(true_labels, predictions)
precision, recall, f1, _ = precision_recall_fscore_support(
    true_labels,
    predictions,
    average="binary",
    zero_division=0
)
cm = confusion_matrix(true_labels, predictions)

results_df = pd.DataFrame([
    {
        "model_name": model_name,
        "dataset": "glue/mrpc",
        "split": "validation",
        "num_examples": len(dataset),
        "batch_size": batch_size,
        "max_length": max_length,
        "similarity_threshold": similarity_threshold,
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "device": str(device)
    }
])

print(results_df.to_string(index=False))
print("Confusion Matrix:")
print(cm)


In [ ]:
examples_df = dataset.to_pandas()[["sentence1", "sentence2", "label"]].copy()
examples_df = examples_df.rename(columns={"label": "true_label"})
examples_df["similarity"] = similarities
examples_df["predicted_label"] = predictions
examples_df["correct"] = examples_df["true_label"] == examples_df["predicted_label"]

print(examples_df.head(10).to_string(index=False))

mismatches_df = examples_df[~examples_df["correct"]].copy()
print(f"\nMismatches: {len(mismatches_df)}")
if len(mismatches_df) > 0:
    sample_errors_df = mismatches_df.head(10)
    print(sample_errors_df[["sentence1", "sentence2", "similarity", "true_label", "predicted_label"]].to_string(index=False))


In [ ]:
elapsed_seconds = time.time() - start_time
print(f"Total runtime (seconds): {elapsed_seconds:.2f}")
